In [5]:
import os
import ast
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.loader import DataLoader
from mlp import global_graph_features, GlobalFeatMLP
from utils import label_components_and_successor, crossing_port_roles, build_designA_graph, to_strand_split_graph, add_reverse_edges
from model import EncoderV2, HurdleEncoderV1, collect_nonzero_values, build_value_codec, decode_values, evaluate_hurdle, compute_pos_weight, hurdle_losses

def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(0)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

device: cpu


In [6]:
POLY_KINDS = ['Conway']

Folding utilities loaded (corrected).
  alex_fold_exp0/unfold_exp0: symmetry a_e = a_{-e} around exponent 0
  conway_fold/unfold: even-degree structure for Conway
  detect_symmetry_center: auto-detect which symmetry applies
  determinant_from_poly_dict: scalar target |Delta(-1)|


In [ ]:
from preprocess import prepare_poly_dataset

## Stage 0: Scalar targets

Quick scalar regression benchmarks

In [8]:
POLY_KINDS = ['Conway']

class ScalarMLP(nn.Module):
    def __init__(self, in_dim, hidden=128, depth=3, dropout=0.1, num_relations=10):
        super().__init__()
        self.num_relations = num_relations
        self.mlp = GlobalFeatMLP(in_dim=in_dim, out_dim=1, hidden=hidden, depth=depth, dropout=dropout)

    def forward(self, batch):
        feats = global_graph_features(batch, num_relations=self.num_relations)
        return self.mlp(feats).squeeze(-1)

class ScalarGNN(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.encoder = EncoderV2(in_dim=in_dim)
        self.head = nn.Sequential(nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 1))

    def forward(self, batch):
        return self.head(self.encoder(batch)).squeeze(-1)


def run_scalar_benchmark(kind_data, target_name='determinant', max_epochs=30, patience=10):
    """Train MLP vs GNN on a scalar target. Returns summary DataFrame."""
    train_graphs = kind_data['train_graphs']
    val_graphs = kind_data['val_graphs']

    # Filter to graphs that have valid target
    def has_target(g):
        t = getattr(g, target_name, None)
        if t is None:
            return False
        return not torch.isnan(t).any()

    tr = [g for g in train_graphs if has_target(g)]
    va = [g for g in val_graphs if has_target(g)]

    if len(tr) < 50 or len(va) < 10:
        print(f"  Skipping {target_name}: only {len(tr)} train / {len(va)} val graphs with valid target")
        return None

    print(f"  {target_name}: {len(tr)} train, {len(va)} val graphs")

    # Attach scalar target as y_scalar
    def attach_target(graphs):
        out = []
        for g in graphs:
            h = g.clone()
            h.y_scalar = getattr(h, target_name).float().view(1)
            out.append(h)
        return out

    tr = attach_target(tr)
    va = attach_target(va)
    tr_loader = DataLoader(tr, batch_size=64, shuffle=True)
    va_loader = DataLoader(va, batch_size=128, shuffle=False)

    # Compute normalisation from train
    with torch.no_grad():
        ys = torch.cat([b.y_scalar.view(-1) for b in tr_loader])
        y_mean = ys.mean()
        y_std = ys.std().clamp(min=1e-6)

    # Log-transform for determinant (always positive, can be large)
    use_log = (target_name == 'determinant')
    if use_log:
        print(f"  Using log1p transform for {target_name}")

    def get_target(batch):
        t = batch.y_scalar.float()
        if use_log:
            t = torch.log1p(t)
        return t

    # Recompute normalisation with transform
    if use_log:
        with torch.no_grad():
            ys = torch.cat([torch.log1p(b.y_scalar.view(-1).float()) for b in tr_loader])
            y_mean = ys.mean()
            y_std = ys.std().clamp(min=1e-6)

    @torch.no_grad()
    def eval_model(model):
        model.eval()
        preds, trues = [], []
        for b in va_loader:
            b = b.to(DEVICE)
            p = model(b)
            # Unnormalise
            p_un = p * y_std.to(DEVICE) + y_mean.to(DEVICE)
            if use_log:
                p_un = torch.expm1(p_un)
            t = batch_raw_target(b)
            preds.append(p_un.cpu())
            trues.append(t.cpu())
        P = torch.cat(preds)
        T = torch.cat(trues)
        mae = (P - T).abs().mean().item()
        rmse = ((P - T) ** 2).mean().sqrt().item()
        # Relative MAE
        rel_mae = ((P - T).abs() / T.abs().clamp(min=1.0)).mean().item()
        return {'mae': mae, 'rmse': rmse, 'rel_mae': rel_mae}

    def batch_raw_target(batch):
        t = batch.y_scalar.float()
        return t

    def train_model(model, lr=1e-3):
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        best = None
        bad = 0
        for ep in range(1, max_epochs + 1):
            model.train()
            for b in tr_loader:
                b = b.to(DEVICE)
                t = get_target(b)
                t_norm = (t - y_mean.to(DEVICE)) / y_std.to(DEVICE)
                opt.zero_grad()
                p = model(b)
                loss = F.smooth_l1_loss(p, t_norm)
                loss.backward()
                opt.step()
            m = eval_model(model)
            if best is None or m['mae'] < best['mae']:
                best = {'epoch': ep, **m}
                bad = 0
            else:
                bad += 1
                if bad >= patience:
                    break
        return best

    # Infer feature dim
    _tmp = next(iter(tr_loader))
    feat_dim = int(global_graph_features(_tmp, num_relations=10).size(1))

    results = []

    set_seed(0)
    mlp = ScalarMLP(in_dim=feat_dim).to(DEVICE)
    best_mlp = train_model(mlp)
    results.append({'poly_kind': kind_data['poly_kind'], 'target': target_name,
                    'model': 'MLP', **best_mlp})

    set_seed(0)
    gnn = ScalarGNN(in_dim=train_graphs[0].x.size(1)).to(DEVICE)
    best_gnn = train_model(gnn)
    results.append({'poly_kind': kind_data['poly_kind'], 'target': target_name,
                    'model': 'GNN', **best_gnn})

    improve = 100.0 * (best_mlp['mae'] - best_gnn['mae']) / max(best_mlp['mae'], 1e-9)
    print(f"  {target_name}: MLP MAE={best_mlp['mae']:.4f}, GNN MAE={best_gnn['mae']:.4f}, "
          f"GNN improvement={improve:.1f}%")

    return pd.DataFrame(results)


# Run scalar benchmarks
print("=" * 60)
print("STAGE 0: Scalar target benchmarks")
print("=" * 60)

scalar_tables = []
poly_datasets = {}

for kind in POLY_KINDS:
    print(f"\nPreparing {kind} dataset...")
    kd = prepare_poly_dataset(kind, use_folding=True)
    poly_datasets[kind] = kd

    print(f"\n--- {kind}: Determinant ---")
    t = run_scalar_benchmark(kd, target_name='determinant')
    if t is not None:
        scalar_tables.append(t)

    if kd['has_signature']:
        print(f"\n--- {kind}: Signature ---")
        t = run_scalar_benchmark(kd, target_name='signature')
        if t is not None:
            scalar_tables.append(t)

if scalar_tables:
    scalar_table = pd.concat(scalar_tables, ignore_index=True)
    print("\n" + "=" * 60)
    print("Stage 0 summary:")
    print(scalar_table.to_string(index=False))
else:
    print("No scalar benchmarks completed.")

STAGE 0: Scalar target benchmarks

Preparing Conway dataset...
[Conway] range=[0,12] L_full=13
  has_signature=False
  determinant range: 0 to 233
  Even-degree verification: 100.0% of odd-exponent entries are zero
  -> Conway folded: L=13 -> L_folded=7 (even-exponent only)
  -> Roundtrip max error: 0.000000
  Building 12965 graphs...
  Cached to alexconway_conway_0_12_conway_even.pt

--- Conway: Determinant ---
  determinant: 11668 train, 1297 val graphs
  Using log1p transform for determinant
  determinant: MLP MAE=5.7783, GNN MAE=4.1466, GNN improvement=28.2%

Stage 0 summary:
poly_kind      target model  epoch      mae     rmse  rel_mae
   Conway determinant   MLP     26 5.778283 8.416270 1.289507
   Conway determinant   GNN     30 4.146621 6.295585 0.916636


## Stage A: Summary targets (nz_count, l1, span, max_abs)

Now with 20 epochs / patience 8 for fairer GNN convergence. Uses folded targets.

In [10]:
def stageA_summary_target(y):
    nz = (y != 0)
    nz_count = nz.float().sum()
    l1 = y.abs().sum()
    max_abs = y.abs().max()
    if nz.any():
        idx = torch.where(nz)[0]
        span = (idx.max() - idx.min()).float()
    else:
        span = torch.tensor(0.0)
    return torch.stack([nz_count, l1, span, max_abs], dim=0)

def attach_stageA_targets(graphs):
    out = []
    for g in graphs:
        h = g.clone()
        # Use folded y for summary (reflects what the model will see)
        y = h.y.view(-1)
        h.y_stageA = stageA_summary_target(y).view(1, -1)
        out.append(h)
    return out

class StageAPolyMLP(nn.Module):
    def __init__(self, in_dim, out_dim=4, hidden=128, depth=3, dropout=0.1, num_relations=10):
        super().__init__()
        self.num_relations = num_relations
        self.mlp = GlobalFeatMLP(in_dim=in_dim, out_dim=out_dim, hidden=hidden, depth=depth, dropout=dropout)

    def forward(self, batch):
        feats = global_graph_features(batch, num_relations=self.num_relations)
        return self.mlp(feats)

class StageAPolyGNN(nn.Module):
    def __init__(self, in_dim, out_dim=4):
        super().__init__()
        self.encoder = EncoderV2(in_dim=in_dim)
        self.head = nn.Sequential(nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, out_dim))

    def forward(self, batch):
        return self.head(self.encoder(batch))

def train_stageA_once(kind_data, max_epochs=20, patience=8):
    tr = attach_stageA_targets(kind_data['train_graphs'])
    va = attach_stageA_targets(kind_data['val_graphs'])
    tr_loader = DataLoader(tr, batch_size=64, shuffle=True)
    va_loader = DataLoader(va, batch_size=128, shuffle=False)

    with torch.no_grad():
        Ys = []
        for b in tr_loader:
            Ys.append(b.y_stageA.view(b.num_graphs, -1))
        Y = torch.cat(Ys, dim=0)
        mean = Y.mean(dim=0)
        std = Y.std(dim=0).clamp(min=1e-6)

    def norm(x):
        return (x - mean.to(x.device)) / std.to(x.device)

    _tmp_batch = next(iter(tr_loader))
    feat_dim = int(global_graph_features(_tmp_batch, num_relations=10).size(1))

    @torch.no_grad()
    def eval_model(model):
        model.eval()
        preds, trues = [], []
        for b in va_loader:
            b = b.to(DEVICE)
            p = model(b)
            p = p * std.to(DEVICE) + mean.to(DEVICE)
            y = b.y_stageA.view(b.num_graphs, -1)
            preds.append(p.cpu())
            trues.append(y.cpu())
        P = torch.cat(preds, dim=0)
        T = torch.cat(trues, dim=0)
        E = (P - T).abs()
        return {
            'mae_per_target': E.mean(dim=0).numpy(),
            'mae_mean': float(E.mean().item()),
            'rmse_mean': float(torch.sqrt(((P - T) ** 2).mean()).item()),
        }

    def train_model(model):
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        best = None
        bad = 0
        for ep in range(1, max_epochs + 1):
            model.train()
            for b in tr_loader:
                b = b.to(DEVICE)
                y = b.y_stageA.view(b.num_graphs, -1)
                opt.zero_grad()
                p = model(b)
                loss = F.smooth_l1_loss(p, norm(y))
                loss.backward()
                opt.step()
            va_m = eval_model(model)
            if best is None or va_m['mae_mean'] < best['mae_mean']:
                best = {'epoch': ep, **va_m}
                bad = 0
            else:
                bad += 1
                if bad >= patience:
                    break
        return best

    set_seed(0)
    mlp = StageAPolyMLP(in_dim=feat_dim, out_dim=4, num_relations=10).to(DEVICE)
    best_mlp = train_model(mlp)

    set_seed(0)
    gnn = StageAPolyGNN(in_dim=kind_data['train_graphs'][0].x.size(1)).to(DEVICE)
    best_gnn = train_model(gnn)

    names = ['nz_count', 'l1', 'span', 'max_abs']
    rows = []
    for i, n in enumerate(names):
        a = float(best_mlp['mae_per_target'][i])
        b = float(best_gnn['mae_per_target'][i])
        rows.append({'poly_kind': kind_data['poly_kind'], 'target': n,
                     'MLP_MAE': a, 'GNN_MAE': b,
                     'improve_%': 100.0 * (a - b) / max(a, 1e-9)})
    return pd.DataFrame(rows), pd.DataFrame([
        {'poly_kind': kind_data['poly_kind'], 'model': 'MLP',
         'best_epoch': best_mlp['epoch'], 'val_MAE_mean': best_mlp['mae_mean'],
         'val_RMSE_mean': best_mlp['rmse_mean']},
        {'poly_kind': kind_data['poly_kind'], 'model': 'EncoderV2_GNN',
         'best_epoch': best_gnn['epoch'], 'val_MAE_mean': best_gnn['mae_mean'],
         'val_RMSE_mean': best_gnn['rmse_mean']},
    ])


print("=" * 60)
print("STAGE A: Summary targets (20 epochs, patience 8)")
print("=" * 60)

stageA_target_tables = []
stageA_compact_tables = []

for kind in POLY_KINDS:
    kd = poly_datasets[kind]
    print(f"\n--- {kind} (L_folded={kd['L']}, L_full={kd['L_full']}) ---")
    t1, t2 = train_stageA_once(kd, max_epochs=20, patience=8)
    stageA_target_tables.append(t1)
    stageA_compact_tables.append(t2)

stageA_target_table = pd.concat(stageA_target_tables, ignore_index=True)
stageA_compact_table = pd.concat(stageA_compact_tables, ignore_index=True)

print("\nStage A per-target results:")
print(stageA_target_table.to_string(index=False))
print("\nStage A compact summary:")
print(stageA_compact_table.sort_values(['poly_kind', 'val_MAE_mean']).to_string(index=False))

STAGE A: Summary targets (20 epochs, patience 8)

--- Conway (L_folded=7, L_full=13) ---

Stage A per-target results:
poly_kind   target  MLP_MAE  GNN_MAE  improve_%
   Conway nz_count 0.693181 0.544726  21.416582
   Conway       l1 4.834043 4.031237  16.607340
   Conway     span 0.576676 0.379916  34.119659
   Conway  max_abs 2.017271 1.615604  19.911395

Stage A compact summary:
poly_kind         model  best_epoch  val_MAE_mean  val_RMSE_mean
   Conway EncoderV2_GNN          20      1.642871       3.194075
   Conway           MLP          18      2.030293       3.866306


## Stage B: Coefficient prediction

Evaluation computes metrics on both folded and unfolded targets for comparison.

In [ ]:
from preprocess import conway_unfold
@torch.no_grad()
def evaluate_hurdle_with_unfold(model, loader, device, kind_data,
                                 value_mode='regression', codec=None, decode_mode='expected'):
    """Evaluate hurdle model, reporting metrics on both folded and unfolded targets."""
    folded_metrics = evaluate_hurdle(model, loader, device, value_mode=value_mode,
                                      codec=codec, decode_mode=decode_mode)

    fold_info = kind_data.get('fold_info', {})
    fold_type = fold_info.get('type', 'none')

    if fold_type in ('none', 'none_support_palindromic'):
        folded_metrics['unfold_val_MAE_nz_soft'] = folded_metrics['val_MAE_nz_soft']
        folded_metrics['unfold_val_mae_values_on_true_nz'] = folded_metrics['val_mae_values_on_true_nz']
        return folded_metrics

    model.eval()
    all_pred_full, all_y_full = [], []

    for b in loader:
        b = b.to(device)
        logits, value_raw = model(b)
        p = torch.sigmoid(logits)
        values_pred = decode_values(value_raw, value_mode=value_mode, codec=codec, decode_mode=decode_mode)
        pred_folded = p * values_pred
        B_size = b.num_graphs

        # Unfold predictions back to full vector
        if fold_type == 'conway_even':
            L_full = fold_info['L_full']
            even_indices = fold_info['even_indices']
            pred_full = conway_unfold(pred_folded, L_full, even_indices)
        else:
            pred_full = pred_folded

        y_full = b.y_full.float().view(B_size, -1) if hasattr(b, 'y_full') else b.y.float().view(B_size, -1)

        all_pred_full.append(pred_full.detach().cpu())
        all_y_full.append(y_full.detach().cpu())

    PRED = torch.cat(all_pred_full, dim=0)
    Y = torch.cat(all_y_full, dim=0)

    nz = (Y != 0)
    mae_nz = float((PRED[nz] - Y[nz]).abs().mean().item()) if nz.any() else float('nan')
    mae_all = float((PRED - Y).abs().mean().item())

    folded_metrics['unfold_val_MAE_nz_soft'] = mae_nz
    folded_metrics['unfold_val_MAE_all_soft'] = mae_all

    return folded_metrics


def run_stageB_poly(kind_data, K, lambda_max, max_epochs=25, patience=6):
    train_loader = kind_data['train_loader']
    val_loader = kind_data['val_loader']
    L_local = kind_data['L']  # folded dim

    pos_weight = compute_pos_weight(train_loader, L_local).to(DEVICE)

    vals = collect_nonzero_values(train_loader)
    max_abs = int(max(abs(v) for v in vals)) if len(vals) else int(K)
    codec = build_value_codec(K=K, max_abs=max_abs, use_overflow=True)
    num_classes = len(codec['class_values'])

    def fit_with_lr(lr):
        set_seed(0)
        model = HurdleEncoderV1(
            in_dim=kind_data['train_graphs'][0].x.size(1),
            out_dim=L_local,
            value_mode='classification',
            num_value_classes=num_classes,
        ).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=lr)

        best = None
        bad = 0
        for ep in range(1, max_epochs + 1):
            model.train()
            lam = lambda_max * min(1.0, ep / 10.0)

            for b in train_loader:
                b = b.to(DEVICE)
                y = b.y.float().view(b.num_graphs, -1)
                opt.zero_grad()
                logits, value_raw = model(b)
                total, bce, reg = hurdle_losses(
                    logits, value_raw, y, pos_weight, lam=lam,
                    value_mode='classification', codec=codec,
                    decode_mode='expected', window_pad=None,
                    teacher_forcing_reg=False,
                )
                total.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

            va = evaluate_hurdle_with_unfold(
                model, val_loader, DEVICE, kind_data,
                value_mode='classification', codec=codec, decode_mode='expected',
            )
            row = {
                'poly_kind': kind_data['poly_kind'],
                'lr': lr,
                'LAMBDA_MAX': lambda_max,
                'K': K,
                'L_folded': L_local,
                'L_full': kind_data['L_full'],
                'fold_type': kind_data['fold_info'].get('type', 'none'),
                'best_epoch': ep,
                **va,
            }

            if best is None or va['val_mae_values_on_true_nz'] < best['val_mae_values_on_true_nz']:
                best = row
                bad = 0
            else:
                bad += 1
                if ep >= 12 and bad >= patience:
                    break

        return best

    return fit_with_lr


print("=" * 60)
print("STAGE B: Coefficient prediction (with symmetry folding)")
print("=" * 60)

stageB_rows = []
for kind in POLY_KINDS:
    kd = poly_datasets[kind]
    fi = kd['fold_info']
    print(f"\n--- {kind}: fold_type={fi.get('type','none')}, "
          f"L_folded={kd['L']}, L_full={kd['L_full']} ---")

    for K in [5, 7]:
        for LMAX in [1, 2]:
            fit_fn = run_stageB_poly(kd, K=K, lambda_max=LMAX, max_epochs=25, patience=6)
            for lr in [3e-4, 1e-3]:
                row = fit_fn(lr)
                stageB_rows.append(row)
                print(f"  K={K} lam={LMAX} lr={lr}: "
                      f"val_mae_nz={row['val_mae_values_on_true_nz']:.4f} "
                      f"unfold_nz={row.get('unfold_val_MAE_nz_soft', 'N/A')}")

stageB_table_poly = pd.DataFrame(stageB_rows).sort_values('val_mae_values_on_true_nz').reset_index(drop=True)

show_cols = [
    'poly_kind', 'fold_type', 'L_folded', 'lr', 'LAMBDA_MAX', 'K', 'best_epoch',
    'val_mae_values_on_true_nz', 'val_MAE_nz_oracle_gate',
    'mask_auc', 'mask_f1', 'best_thresh',
    'val_MAE_nz_soft', 'val_MAE_all_soft',
    'unfold_val_MAE_nz_soft',
]
show_cols = [c for c in show_cols if c in stageB_table_poly.columns]

print("\n" + "=" * 60)
print("Stage B results (sorted by val_mae_values_on_true_nz):")
print(stageB_table_poly[show_cols].to_string(index=False))

STAGE B: Coefficient prediction (with symmetry folding)

--- Conway: fold_type=conway_even, L_folded=7, L_full=13 ---
  K=5 lam=1 lr=0.0003: val_mae_nz=1.5735 unfold_nz=1.603109359741211
  K=5 lam=1 lr=0.001: val_mae_nz=1.3855 unfold_nz=1.3958590030670166
  K=5 lam=2 lr=0.0003: val_mae_nz=1.5443 unfold_nz=1.5563713312149048
  K=5 lam=2 lr=0.001: val_mae_nz=1.4559 unfold_nz=1.4642473459243774
  K=7 lam=1 lr=0.0003: val_mae_nz=1.5827 unfold_nz=1.609230637550354
  K=7 lam=1 lr=0.001: val_mae_nz=1.3061 unfold_nz=1.3185083866119385
  K=7 lam=2 lr=0.0003: val_mae_nz=1.5702 unfold_nz=1.5971307754516602
  K=7 lam=2 lr=0.001: val_mae_nz=1.3094 unfold_nz=1.3164414167404175

Stage B results (sorted by val_mae_values_on_true_nz):
poly_kind   fold_type  L_folded     lr  LAMBDA_MAX  K  best_epoch  val_mae_values_on_true_nz  val_MAE_nz_oracle_gate  mask_auc  mask_f1  best_thresh  val_MAE_nz_soft  val_MAE_all_soft  unfold_val_MAE_nz_soft
   Conway conway_even         7 0.0010           1  7          2

In [ ]:
# --- Final summary ---
print("=" * 60)
print("EXPERIMENT SUMMARY")
print("=" * 60)

print("\n--- Stage 0: Scalar targets ---")
if scalar_tables:
    print(pd.concat(scalar_tables, ignore_index=True).to_string(index=False))

print("\n--- Stage A: Summary targets ---")
print(stageA_target_table.to_string(index=False))

print("\n--- Stage B: Best per polynomial kind ---")
for kind in POLY_KINDS:
    sub = stageB_table_poly[stageB_table_poly['poly_kind'] == kind]
    if len(sub) > 0:
        best = sub.iloc[0]
        print(f"\n  {kind} (fold={best.get('fold_type','none')}, L_folded={best.get('L_folded','?')}):")
        print(f"    val_mae_values_on_true_nz = {best['val_mae_values_on_true_nz']:.4f}")
        print(f"    mask_auc = {best['mask_auc']:.4f}")
        print(f"    val_MAE_nz_soft (folded) = {best['val_MAE_nz_soft']:.4f}")
        if 'unfold_val_MAE_nz_soft' in best:
            print(f"    unfold_val_MAE_nz_soft (full) = {best['unfold_val_MAE_nz_soft']:.4f}")

print("\n--- Folding statistics ---")
for kind in POLY_KINDS:
    kd = poly_datasets[kind]
    fi = kd['fold_info']
    print(f"  {kind}: {fi.get('type','none')} | L_full={kd['L_full']} -> L_folded={kd['L']} "
          f"({100*(1-kd['L']/kd['L_full']):.0f}% reduction)")

EXPERIMENT SUMMARY

--- Stage 0: Scalar targets ---
poly_kind      target model  epoch      mae     rmse  rel_mae
   Conway determinant   MLP     30 4.831932 7.211115 1.003913
   Conway determinant   GNN     28 2.441924 3.647007 0.507999

--- Stage A: Summary targets ---
poly_kind   target  MLP_MAE  GNN_MAE  improve_%
   Conway nz_count 0.674251 0.448833  33.432302
   Conway       l1 4.275733 2.396416  43.953096
   Conway     span 0.576674 0.260161  54.885948
   Conway  max_abs 1.744822 1.063746  39.034136

--- Stage B: Best per polynomial kind ---

  Conway (fold=conway_even, L_folded=7):
    val_mae_values_on_true_nz = 1.3061
    mask_auc = 0.9819
    val_MAE_nz_soft (folded) = 1.3185
    unfold_val_MAE_nz_soft (full) = 1.3185

--- Folding statistics ---
  Conway: conway_even | L_full=13 -> L_folded=7 (46% reduction)


In [ ]:
# --- Stage B MLP baseline (append after existing notebook) ---
# Reuses poly_datasets and stageB_table_poly already in memory.

class HurdleMLP(nn.Module):
    def __init__(self, feat_dim, out_dim, value_mode='regression',
                 num_value_classes=None, hidden=128, num_relations=10):
        super().__init__()
        self.value_mode = value_mode
        self.num_relations = num_relations
        self.trunk = nn.Sequential(
            nn.Linear(feat_dim, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.1),
        )
        self.mask_head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                       nn.Linear(hidden, out_dim))
        if value_mode == 'classification':
            assert num_value_classes is not None
            self.value_head = nn.Sequential(
                nn.Linear(hidden, hidden), nn.ReLU(),
                nn.Linear(hidden, out_dim * num_value_classes))
            self.num_value_classes = num_value_classes
            self.out_dim = out_dim
        else:
            self.value_head = nn.Sequential(
                nn.Linear(hidden, hidden), nn.ReLU(),
                nn.Linear(hidden, out_dim))

    def forward(self, batch):
        feats = global_graph_features(batch, num_relations=self.num_relations)
        h = self.trunk(feats)
        logits = self.mask_head(h)
        value_raw = self.value_head(h)
        if self.value_mode == 'classification':
            value_raw = value_raw.view(-1, self.out_dim, self.num_value_classes)
        return logits, value_raw


mlp_rows = []
for kind in POLY_KINDS:
    kd = poly_datasets[kind]
    L_local = kd['L']

    _tmp = next(iter(kd['train_loader']))
    feat_dim = int(global_graph_features(_tmp, num_relations=10).size(1))

    pos_weight = compute_pos_weight(kd['train_loader'], L_local).to(DEVICE)
    vals = collect_nonzero_values(kd['train_loader'])
    max_abs = int(max(abs(v) for v in vals)) if len(vals) else 5

    for K in [5, 7]:
        codec = build_value_codec(K=K, max_abs=max_abs, use_overflow=True)
        num_classes = len(codec['class_values'])

        for LMAX in [1, 2]:
            for lr in [3e-4, 1e-3]:
                set_seed(0)
                model = HurdleMLP(
                    feat_dim=feat_dim, out_dim=L_local,
                    value_mode='classification',
                    num_value_classes=num_classes,
                ).to(DEVICE)
                opt = torch.optim.Adam(model.parameters(), lr=lr)

                best = None
                bad = 0
                for ep in range(1, 26):
                    model.train()
                    lam = LMAX * min(1.0, ep / 10.0)
                    for b in kd['train_loader']:
                        b = b.to(DEVICE)
                        y = b.y.float().view(b.num_graphs, -1)
                        opt.zero_grad()
                        logits, value_raw = model(b)
                        total, bce, reg = hurdle_losses(
                            logits, value_raw, y, pos_weight, lam=lam,
                            value_mode='classification', codec=codec,
                            decode_mode='expected',
                        )
                        total.backward()
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        opt.step()

                    va = evaluate_hurdle(
                        model, kd['val_loader'], DEVICE,
                        value_mode='classification', codec=codec,
                        decode_mode='expected',
                    )
                    row = {'poly_kind': kind, 'model': 'MLP',
                           'lr': lr, 'LAMBDA_MAX': LMAX, 'K': K,
                           'best_epoch': ep, **va}
                    if best is None or va['val_mae_values_on_true_nz'] < best['val_mae_values_on_true_nz']:
                        best = row
                        bad = 0
                    else:
                        bad += 1
                        if ep >= 12 and bad >= 6:
                            break

                mlp_rows.append(best)
                print(f"  {kind} K={K} lam={LMAX} lr={lr}: "
                      f"MLP val_mae_nz={best['val_mae_values_on_true_nz']:.4f}")

mlp_table = pd.DataFrame(mlp_rows)

# --- Comparison: best GNN vs best MLP per polynomial kind ---
stageB_table_poly['model'] = 'GNN'
combined = pd.concat([stageB_table_poly, mlp_table], ignore_index=True)

print("\n" + "=" * 60)
print("Stage B: Best GNN vs Best MLP per polynomial kind")
print("=" * 60)

for kind in POLY_KINDS:
    sub = combined[combined['poly_kind'] == kind]
    best_gnn = sub[sub['model'] == 'GNN'].sort_values('val_mae_values_on_true_nz').iloc[0]
    best_mlp = sub[sub['model'] == 'MLP'].sort_values('val_mae_values_on_true_nz').iloc[0]

    improve_val = 100.0 * (best_mlp['val_mae_values_on_true_nz'] - best_gnn['val_mae_values_on_true_nz']) \
                  / max(best_mlp['val_mae_values_on_true_nz'], 1e-9)
    improve_mask = 100.0 * (best_mlp['mask_auc'] - best_gnn['mask_auc']) \
                   / max(abs(best_mlp['mask_auc']), 1e-9)

    print(f"\n{kind}:")
    print(f"  val_mae_values_on_true_nz:  GNN={best_gnn['val_mae_values_on_true_nz']:.4f}  "
          f"MLP={best_mlp['val_mae_values_on_true_nz']:.4f}  GNN improve={improve_val:+.1f}%")
    print(f"  mask_auc:                   GNN={best_gnn['mask_auc']:.4f}  "
          f"MLP={best_mlp['mask_auc']:.4f}")
    print(f"  mask_f1:                    GNN={best_gnn['mask_f1']:.4f}  "
          f"MLP={best_mlp['mask_f1']:.4f}")
    print(f"  val_MAE_nz_soft:            GNN={best_gnn['val_MAE_nz_soft']:.4f}  "
          f"MLP={best_mlp['val_MAE_nz_soft']:.4f}")

  Conway K=5 lam=1 lr=0.0003: MLP val_mae_nz=2.1565
  Conway K=5 lam=1 lr=0.001: MLP val_mae_nz=1.9600
  Conway K=5 lam=2 lr=0.0003: MLP val_mae_nz=2.1689
  Conway K=5 lam=2 lr=0.001: MLP val_mae_nz=1.9603
  Conway K=7 lam=1 lr=0.0003: MLP val_mae_nz=2.1431
  Conway K=7 lam=1 lr=0.001: MLP val_mae_nz=1.9058
  Conway K=7 lam=2 lr=0.0003: MLP val_mae_nz=2.1275
  Conway K=7 lam=2 lr=0.001: MLP val_mae_nz=1.8903

Stage B: Best GNN vs Best MLP per polynomial kind

Conway:
  val_mae_values_on_true_nz:  GNN=1.3061  MLP=1.8903  GNN improve=+30.9%
  mask_auc:                   GNN=0.9819  MLP=0.9421
  mask_f1:                    GNN=0.9598  MLP=0.9053
  val_MAE_nz_soft:            GNN=1.3185  MLP=1.9332


In [ ]:
import torch
from torch_geometric.data import Batch

best_row = stageB_table_poly[stageB_table_poly['poly_kind'] == 'Conway'] \
    .sort_values('val_mae_values_on_true_nz') \
    .iloc[0].to_dict()

print("Best Conway config:", best_row)

kd = poly_datasets['Conway']
L_folded = int(kd['L'])
in_dim = int(kd['train_graphs'][0].x.size(1))

# Rebuild codec
vals = collect_nonzero_values(kd['train_loader'])
max_abs = int(max(abs(v) for v in vals)) if len(vals) else int(best_row['K'])
codec = build_value_codec(K=int(best_row['K']), max_abs=max_abs, use_overflow=True)
num_classes = len(codec['class_values'])

pos_weight = compute_pos_weight(kd['train_loader'], L_folded).to(DEVICE)

# Train a fresh model with the chosen hyperparams
set_seed(0)
model_best = HurdleEncoderV1(
    in_dim=in_dim,
    out_dim=L_folded,
    value_mode='classification',
    num_value_classes=num_classes,
).to(DEVICE)

opt = torch.optim.Adam(model_best.parameters(), lr=float(best_row['lr']))

best_state = None
best_metric = float('inf')

max_epochs = int(best_row.get('best_epoch', 25))
lambda_max = float(best_row['LAMBDA_MAX'])

for ep in range(1, max_epochs + 1):
    model_best.train()
    lam = lambda_max * min(1.0, ep / 10.0)

    for b in kd['train_loader']:
        b = b.to(DEVICE)
        y = b.y.float().view(b.num_graphs, -1)

        opt.zero_grad()
        logits, value_raw = model_best(b)
        total, bce, reg = hurdle_losses(
            logits, value_raw, y,
            pos_weight=pos_weight, lam=lam,
            value_mode='classification', codec=codec,
            decode_mode='expected',
            window_pad=None,
            teacher_forcing_reg=False,
        )
        total.backward()
        torch.nn.utils.clip_grad_norm_(model_best.parameters(), 1.0)
        opt.step()

    va = evaluate_hurdle_with_unfold(
        model_best, kd['val_loader'], DEVICE, kd,
        value_mode='classification', codec=codec, decode_mode='expected',
    )

    if va['val_mae_values_on_true_nz'] < best_metric:
        best_metric = va['val_mae_values_on_true_nz']
        best_state = {k: v.detach().cpu() for k, v in model_best.state_dict().items()}

print("Best val_mae_values_on_true_nz during save-train:", best_metric)

Best Conway config: {'poly_kind': 'Conway', 'lr': 0.001, 'LAMBDA_MAX': 1, 'K': 7, 'L_folded': 7, 'L_full': 13, 'fold_type': 'conway_even', 'best_epoch': 24, 'val_MAE_all_soft': 0.862113356590271, 'val_MAE_nz_soft': 1.318508505821228, 'val_MAE_nz_hard': 1.3064422607421875, 'val_MAE_nz_oracle_gate': 1.3060879707336426, 'val_mae_values_on_true_nz': 1.3060879707336426, 'mask_auc': 0.9819178493894817, 'mask_f1': 0.9597602739726028, 'best_thresh': 0.55, 'gate_fn_rate': 0.006557958170861397, 'gate_fp_rate': 0.1259819610125109, 'val_pred_abs_mean': 1.401741623878479, 'val_value_exact_acc_nz': 0.4351293742656708, 'unfold_val_MAE_nz_soft': 1.3185083866119385, 'unfold_val_MAE_all_soft': 0.46421489119529724, 'model': 'GNN'}
Best val_mae_values_on_true_nz during save-train: 1.358871579170227


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

ckpt_path = '/content/drive/MyDrive/conway_stageB_best_deep.pt'
torch.save({
    'state_dict': best_state,
    'config': best_row,
    'codec': codec,
    'fold_info': kd.get('fold_info', {}),
    'in_dim': in_dim,
    'out_dim': L_folded,
    'num_value_classes': num_classes,
}, ckpt_path)

print("Saved checkpoint to:", ckpt_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved checkpoint to: /content/drive/MyDrive/conway_stageB_best_deep_enriched.pt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ckpt_path = '/content/drive/MyDrive/conway_stageB_best_deep.pt'

In [ ]:
ckpt = torch.load('conway_stageB_best_deep.pt', map_location=DEVICE)
codec = ckpt["codec"]

# 2) rebuild the model object (same class + args as training)
skein_model = HurdleEncoderV1(
    in_dim=ckpt["in_dim"],
    out_dim=ckpt["out_dim"],
    value_mode="classification",
    num_value_classes=ckpt["num_value_classes"],
).to(DEVICE)

# 3) load weights
skein_model.load_state_dict(ckpt["state_dict"])
skein_model.eval()

/var/folders/jl/7s4khfr57w5cd771qs147z380000gn/T/ipykernel_85765/1713492922.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load('../conway_stageB_best_deep

HurdleEncoderV1(
  (encoder): EncoderV2(
    (input_proj): Linear(in_features=3, out_features=128, bias=True)
    (convs): ModuleList(
      (0-5): 6 x RGCNConv(128, 128, num_relations=10)
    )
    (norms): ModuleList(
      (0-5): 6 x LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
  )
  (mask_head): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=7, bias=True)
  )
  (value_head): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=112, bias=True)
  )
)

In [ ]:
from interp import build_graph_for_inference, skein_explain, flip_crossing_pd, EmbeddingIntervenor, pred_from_outputs, node_ablation_attribution, find_minimal_subgraph, SingleGraphLayerExtractor

import pandas as pd
from collections import defaultdict
import numpy as np

print("\n--- Final-layer node ablation ---\n")

ablation_records = []
skein_records = []

for si in range(min(30, len(kd['val_graphs']))):
    row = kd['val_df'].iloc[si]
    pd_list = row['pd_list']
    n = len(pd_list)
    y_folded = kd['val_graphs'][si].y.view(-1)

    # Ablation attribution (model-internal)
    abl_attr, pred = node_ablation_attribution(
        skein_model, pd_list, DEVICE, kd, codec)

    # Skein attribution (sensitivity) for comparison
    expl = skein_explain(
        skein_model, pd_list, DEVICE, kd,
        value_mode='classification', codec=codec,
        y_true_folded=y_folded)
    skein_attr = expl['attributions']

    for ci in range(n):
        ablation_records.append({
            'knot_idx': si,
            'crossing_idx': ci,
            'ablation_attr': abl_attr[ci].item(),
            'skein_attr': skein_attr[ci].item(),
            'n_crossings': n,
        })

adf = pd.DataFrame(ablation_records)

# Correlation between methods
r = adf['ablation_attr'].corr(adf['skein_attr'])
print(f"Correlation(ablation, skein): r = {r:.4f}")

# Per-knot correlation
pk_corrs = []
for si in adf['knot_idx'].unique():
    sub = adf[adf['knot_idx'] == si]
    if sub['ablation_attr'].std() > 1e-8 and sub['skein_attr'].std() > 1e-8:
        pk_corrs.append(sub['ablation_attr'].corr(sub['skein_attr']))
if pk_corrs:
    print(f"\nPer-knot correlation(ablation, skein):")
    print(f"  mean r = {np.mean(pk_corrs):.4f}")
    print(f"  median r = {np.median(pk_corrs):.4f}")
    print(f"  fraction positive: {(np.array(pk_corrs) > 0).mean():.1%}")

# Distribution of ablation attributions
print(f"\nAblation attribution distribution:")
print(f"  mean = {adf['ablation_attr'].mean():.4f}")
print(f"  std = {adf['ablation_attr'].std():.4f}")
print(f"  top-10% mean = {adf.nlargest(int(len(adf)*0.1), 'ablation_attr')['ablation_attr'].mean():.4f}")
print(f"  bottom-10% mean = {adf.nsmallest(int(len(adf)*0.1), 'ablation_attr')['ablation_attr'].mean():.4f}")
print(f"  ratio (top/bottom 10%): {adf.nlargest(int(len(adf)*0.1), 'ablation_attr')['ablation_attr'].mean() / max(adf.nsmallest(int(len(adf)*0.1), 'ablation_attr')['ablation_attr'].mean(), 1e-10):.2f}x")

# ---- Method 3: Minimal subgraph (sample 10 knots) ----
print(f"{'='*60}")
print("--- Method 3: Minimal sufficient subgraph ---")
print(f"{'='*60}\n")

subgraph_fracs = []

for si in range(min(10, len(kd['val_graphs']))):
    row = kd['val_df'].iloc[si]
    pd_list = row['pd_list']
    n = len(pd_list)

    kept, removal_order, pred = find_minimal_subgraph(
        skein_model, pd_list, DEVICE, kd, codec, tolerance=0.5)

    frac = len(kept) / n
    subgraph_fracs.append(frac)

    # Show the removal trajectory
    print(f"Knot {si} ({n} crossings): minimal subgraph = {len(kept)} "
          f"crossings ({frac:.0%})")
    print(f"  Kept: {kept}")
    if len(removal_order) > 0:
        last_removed, _, final_mae = removal_order[-1]
        print(f"  Last removed: crossing {last_removed} (MAE at removal: {final_mae:.4f})")
    print()

if subgraph_fracs:
    print(f"Summary: models needs {np.mean(subgraph_fracs):.0%} of crossings "
          f"on average (tolerance=0.5)")
    print(f"  min: {min(subgraph_fracs):.0%}, max: {max(subgraph_fracs):.0%}")

def predict_folded_vector(model, g, device, value_mode='classification', codec=None, decode_mode='expected'):
    """Returns folded polynomial vector yhat = sigmoid(logits) * decoded_values, shape [L_folded]."""
    model.eval()
    g = g.to(device)
    with torch.no_grad():
        logits, value_raw = model(g)
        p = torch.sigmoid(logits)  # [1, L]
        v = decode_values(value_raw, value_mode=value_mode, codec=codec, decode_mode=decode_mode)  # [1, L]
        y = (p * v).squeeze(0)     # [L]
    return y

def build_delta_dataset(
    model,
    df,                      # kd['train_df'] or kd['val_df']
    device,
    value_mode='classification',
    codec=None,
    n_knots=300,
    crossings_per_knot=4,
    seed=0,
):
    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(df), size=min(n_knots, len(df)), replace=False)

    extractor = SingleGraphLayerExtractor(model)
    layer_X = defaultdict(list)
    Y = []

    for k, row_idx in enumerate(idxs):
        pd_list = df.iloc[row_idx]["pd_list"]
        n = len(pd_list)
        if n == 0:
            continue

        # Pick a few crossings
        cidxs = rng.choice(n, size=min(crossings_per_knot, n), replace=False)

        # Base outputs/embeddings
        emb_base = extractor.get_pooled_embeddings(pd_list, device)
        y_base = predict_folded_vector(model, build_graph_for_inference(pd_list), device,
                                       value_mode=value_mode, codec=codec).cpu().numpy()

        for ci in cidxs:
            pd_flip = flip_crossing_pd(pd_list, ci)

            emb_flip = extractor.get_pooled_embeddings(pd_flip, device)
            y_flip = predict_folded_vector(model, build_graph_for_inference(pd_flip), device,
                                           value_mode=value_mode, codec=codec).cpu().numpy()

            dy = (y_base - y_flip)  # [L]
            Y.append(dy)

            for name in emb_base.keys():
                dh = emb_base[name] - emb_flip[name]  # [d]
                layer_X[name].append(dh)

        if (k + 1) % 50 == 0:
            print(f"  processed {k+1}/{len(idxs)} knots")

    extractor.remove_hooks()

    Y = np.stack(Y, axis=0)
    layer_X = {name: np.stack(v, axis=0) for name, v in layer_X.items()}
    return layer_X, Y

def ridge_fit_predict(X_tr, Y_tr, X_va, lam=1.0):
    X_tr = np.asarray(X_tr)
    X_va = np.asarray(X_va)
    Y_tr = np.asarray(Y_tr)

    # Add bias
    Xtr = np.column_stack([X_tr, np.ones((X_tr.shape[0], 1))])
    Xva = np.column_stack([X_va, np.ones((X_va.shape[0], 1))])

    d = Xtr.shape[1]
    XtX = Xtr.T @ Xtr + lam * np.eye(d)
    XtY = Xtr.T @ Y_tr
    W = np.linalg.solve(XtX, XtY)  # [d, L]

    return Xva @ W
import numpy as np

def active_dims_by_std(Y, eps=1e-4):
    """
    Returns indices j where std(Y[:,j]) >= eps.
    """
    s = np.std(Y, axis=0)
    return [j for j in range(Y.shape[1]) if s[j] >= eps], s

def zscore_targets(Y_tr, Y_va, eps=1e-8):
    """Columnwise z-score using train stats."""
    mu = Y_tr.mean(axis=0, keepdims=True)
    sd = Y_tr.std(axis=0, keepdims=True)
    sd = np.maximum(sd, eps)
    return (Y_tr - mu) / sd, (Y_va - mu) / sd, mu.squeeze(0), sd.squeeze(0)

def r2_per_column(Y_true, Y_pred, eps=1e-8):
    """R^2 per column; returns nan if variance below eps."""
    Y_true = np.asarray(Y_true)
    if len(np.unique(Y_true)) == 1:
        return np.full(Y_true.shape[1], np.nan, dtype=np.float64)
    Y_pred = np.asarray(Y_pred)
    L = Y_true.shape[1]
    r2 = np.full(L, np.nan, dtype=np.float64)
    for j in range(L):
        y = Y_true[:, j]
        if np.std(y) < eps:
            continue
        ss_res = np.sum((y - Y_pred[:, j]) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r2[j] = 1.0 - ss_res / max(ss_tot, 1e-12)
    return r2

def mae_per_column(Y_true, Y_pred):
    return np.mean(np.abs(Y_true - Y_pred), axis=0)

train_X, train_dY = build_delta_dataset(
    skein_model, kd['train_df'], DEVICE,
    value_mode='classification', codec=codec,
    n_knots=400, crossings_per_knot=3, seed=0
)

val_X, val_dY = build_delta_dataset(
    skein_model, kd['val_df'], DEVICE,
    value_mode='classification', codec=codec,
    n_knots=150, crossings_per_knot=3, seed=1
)
layer_names = sorted(train_X.keys())
L_folded = train_dY.shape[1]

active, stds = active_dims_by_std(val_dY, eps=1e-4)
print("Delta y std per coeff:", stds)
print("Active dims:", active, " -> ", [f"z^{2*j}" for j in active])

USE_ZSCORE = True 

print(f"\n--- R^2 (and MAE) per layer for predicting Delta y from Delta h ---")
header = f"{'Layer':<10s}"
for j in range(L_folded):
    header += f"  z^{2*j:<3d}"
header += "   mean(active)"
print(header)
print("-" * len(header))

for name in layer_names:
    Ytr = train_dY
    Yva = val_dY

    if USE_ZSCORE:
        Ytr_z, Yva_z, mu, sd = zscore_targets(Ytr, Yva)
        Y_pred_z = ridge_fit_predict(train_X[name], Ytr_z, val_X[name], lam=5.0)
        # compute R^2 in z-scored space (more comparable across coeffs)
        r2s = r2_per_column(Yva_z, Y_pred_z, eps=1e-8)
        # MAE in original units (invert z-score)
        Y_pred = Y_pred_z * sd + mu
        maes = mae_per_column(Yva, Y_pred)
    else:
        Y_pred = ridge_fit_predict(train_X[name], Ytr, val_X[name], lam=5.0)
        r2s = r2_per_column(Yva, Y_pred, eps=1e-8)
        maes = mae_per_column(Yva, Y_pred)

    row = f"{name:<10s}"
    for r in r2s:
        row += f"  {r:5.3f}" if not np.isnan(r) else "    nan"
    mean_r2 = float(np.nanmean([r2s[j] for j in active])) if active else float('nan')
    row += f"   {mean_r2: .3f}"
    print(row)

    mean_mae = float(np.mean([maes[j] for j in active])) if active else float('nan')
    print(f"{'':<10s}  (mean MAE over active coeffs): {mean_mae:.4f}")


--- Final-layer node ablation ---

Correlation(ablation, skein): r = 0.2848

Per-knot correlation(ablation, skein):
  mean r = -0.0085
  median r = -0.0526
  fraction positive: 43.3%

Ablation attribution distribution:
  mean = 0.2235
  std = 0.2222
  top-10% mean = 0.7390
  bottom-10% mean = 0.0328
  ratio (top/bottom 10%): 22.55x

--- Method 2: Layer-wise ablation ---

Knot 0 (12 crossings):
  conv_0: mean=1.227, max=3.720, top3=[c3=3.72, c7=2.33, c2=1.35]
  conv_1: mean=0.933, max=2.562, top3=[c9=2.56, c7=1.65, c3=1.59]
  conv_2: mean=0.889, max=2.679, top3=[c7=2.68, c9=2.19, c11=1.09]
  conv_3: mean=0.445, max=0.829, top3=[c7=0.83, c8=0.77, c9=0.62]
  conv_4: mean=0.329, max=0.564, top3=[c6=0.56, c7=0.46, c11=0.45]
  conv_5: mean=0.198, max=0.338, top3=[c4=0.34, c6=0.33, c11=0.25]
  Correlation(first_layer, last_layer attribution): r=-0.245

Knot 1 (13 crossings):
  conv_0: mean=1.224, max=2.277, top3=[c3=2.28, c8=2.27, c5=2.06]
  conv_1: mean=0.957, max=2.739, top3=[c2=2.74, c7=1

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

def predict_folded_tensor(model, g, value_mode="classification", codec=None, decode_mode="expected"):
    """Differentiable prediction (no torch.no_grad). Returns tensor [L_folded]."""
    logits, value_raw = model(g)
    p = torch.sigmoid(logits)  # [1, L]
    v = decode_values(value_raw, value_mode=value_mode, codec=codec, decode_mode=decode_mode)  # [1, L]
    return (p * v).squeeze(0)  # [L]

def crossing_to_node_mask(m_cross, N_cross):
    """Expand crossing mask [N] -> node mask [2N, 1] (under + over)."""
    m_node = torch.cat([m_cross, m_cross], dim=0).view(2 * N_cross, 1)
    return m_node

class CrossingMaskExplainer:
    """
    Learns a crossing mask m to preserve a chosen output coefficient.
    Gates intermediate conv outputs (node embeddings) at every conv layer.
    """
    def __init__(self, model):
        self.model = model
        self.hooks = []
        self._active = False
        self._m_node = None  # [2N,1] on device

        enc = model.encoder if hasattr(model, "encoder") else model
        self.convs = enc.convs

    def _hook(self, module, inputs, output):
        if not self._active:
            return None

        # baseline: per-feature mean across nodes (shape [1, hidden])
        b = output.mean(dim=0, keepdim=True)

        # gate deviations from baseline, crossing-wise
        return b + (output - b) * self._m_node

    def __enter__(self):
        # Register hooks once
        for conv in self.convs:
            self.hooks.append(conv.register_forward_hook(self._hook))
        return self

    def __exit__(self, exc_type, exc, tb):
        for h in self.hooks:
            h.remove()
        self.hooks = []

    def explain_one(
        self,
        pd_list,
        coeff_idx,
        device,
        value_mode="classification",
        codec=None,
        n_steps=600,
        lr=5e-2,
        lam_l1=1e-2,
        lam_bin=5e-2,
        target_keep_frac=0.20,
        init_keep=True,
        verbose=False,
    ):
        self.model.eval()
        g = build_graph_for_inference(pd_list).to(device)
        n_nodes = g.x.size(0)
        assert n_nodes % 2 == 0, "Expected strand-split graph with 2N nodes."
        N = n_nodes // 2

        # Target (original prediction)
        with torch.no_grad():
            y0 = predict_folded_tensor(self.model, g, value_mode=value_mode, codec=codec).detach()  # [L]
            y0_j = y0[coeff_idx].detach()

        # Learnable logits -> sigmoid mask in [0,1]
        if init_keep:
            mask_logits = torch.full((N,), 2.0, device=device, requires_grad=True)  # near 1
        else:
            mask_logits = torch.zeros((N,), device=device, requires_grad=True)      # ~0.5

        opt = torch.optim.Adam([mask_logits], lr=lr)

        for step in range(1, n_steps + 1):
            opt.zero_grad()

            m = torch.sigmoid(mask_logits)                      # [N]
            self._m_node = crossing_to_node_mask(m, N)          # [2N,1] on device
            self._active = True

            y = predict_folded_tensor(self.model, g, value_mode=value_mode, codec=codec)  # [L]
            self._active = False

            # Fidelity on the explained coefficient
            # multi-target fidelity 
            if isinstance(coeff_idx, (list, tuple, np.ndarray)):
                idxs = torch.tensor(list(coeff_idx), device=device, dtype=torch.long)
            else:
                idxs = torch.tensor([int(coeff_idx)], device=device, dtype=torch.long)

            y0_slice = y0[idxs]
            y_slice  = y[idxs]
            fid = (y_slice - y0_slice).pow(2).mean()

            # Sparsity + binarisation
            l1 = m.mean()
            bin_pen = (m * (1 - m)).mean()
            keep_pen = (m.mean() - float(target_keep_frac)).pow(2)

            loss = fid + lam_l1 * l1 + lam_bin * bin_pen + 0.1 * keep_pen
            loss.backward()
            opt.step()

            if verbose and (step == 1 or step % 100 == 0 or step == n_steps):
                with torch.no_grad():
                    mae_all = (y - y0).abs().mean().item()
                    print(f"step {step:4d} loss={loss.item():.3e} fid={fid.item():.3e} "
                          f"mean(m)={m.mean().item():.3f} MAE(all)={mae_all:.3e}")

        # Final eval
        with torch.no_grad():
            m = torch.sigmoid(mask_logits).detach()
            self._m_node = crossing_to_node_mask(m, N)
            self._active = True
            y = predict_folded_tensor(self.model, g, value_mode=value_mode, codec=codec).detach()
            self._active = False

            fidelity_abs = float((y_slice - y0_slice).abs().mean().item())
            full_mae = float((y - y0).abs().mean().item())

        return {
            "mask_cross": m.cpu(),
            "N_crossings": N,
            "coeff_idx": idxs,
            "fidelity_abs": fidelity_abs,
            "full_mae": full_mae,
        }

def explain_with_restarts(
    model, pd_list, coeff_idx, device, codec,
    n_restarts=5, seed0=0, **kwargs
):
    """Run explainer multiple times, report stability of top crossings."""
    outs = []
    top_sets = []
    with CrossingMaskExplainer(model) as expl:
        for r in range(n_restarts):
            torch.manual_seed(seed0 + r)
            np.random.seed(seed0 + r)
            out = expl.explain_one(pd_list, coeff_idx, device, codec=codec, **kwargs)
            outs.append(out)

            m = out["mask_cross"]
            k = max(1, int(0.2 * len(m)))  # top 20% by default
            idx = torch.topk(m, k=k).indices.tolist()
            top_sets.append(set(idx))

    # Jaccard stability across restarts
    jac = []
    for i in range(len(top_sets)):
        for j in range(i+1, len(top_sets)):
            A, B = top_sets[i], top_sets[j]

            jac.append(len(A & B) / max(1, len(A | B)))
    return outs, (float(np.mean(jac)) if jac else float("nan"))


Mean Jaccard(top-20%) across restarts: 1.0
One run: fidelity_abs= 0.00038254261016845703 full_mae= 0.2606906294822693
Top crossings: [0, 3, 4, 10, 1, 9, 8, 11, 7, 5, 6, 2]


In [ ]:
import numpy as np
import torch
from tqdm import tqdm
from interp import predict_folded_tensor, crossing_to_node_mask


def pred(pd_list, model=skein_model, device=DEVICE):
    g = build_graph_for_inference(pd_list).to(device)
    model.eval()
    with torch.no_grad():
        y = predict_folded_tensor(model, g, codec=codec).detach().cpu().numpy()
    return y

def eval_mask(model, pd_list, m_cross, device, coeff_idx=None):
    """Evaluate error with a fixed mask using current explainer hooks."""
    g = build_graph_for_inference(pd_list).to(device)
    with torch.no_grad():
        y0 = pred(pd_list)  # numpy
    with CrossingMaskExplainer(model) as expl:
        N = g.x.size(0)//2
        expl._m_node = crossing_to_node_mask(m_cross.to(device), N)
        expl._active = True
        with torch.no_grad():
            y = predict_folded_tensor(model, g, codec=codec).detach().cpu().numpy()
        expl._active = False

    full_mae = float(np.mean(np.abs(y - y0)))
    if coeff_idx is None:
        return full_mae, None
    return full_mae, float(abs(y[coeff_idx] - y0[coeff_idx]))

def random_binary_mask(N, k, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.choice(N, size=k, replace=False)
    m = torch.zeros(N)
    m[idx] = 1.0
    return m

def sweep_keep_frac(pd_list, coeff_idx, fracs=(0.1,0.2,0.3,0.4,0.6,0.8), n_rand=50, seed=0):
    results = []
    N = len(pd_list)

    for f in fracs:
        k = max(1, int(round(f * N)))

        # Learn mask at this target_keep_frac
        outs, _ = explain_with_restarts(
            skein_model, pd_list, coeff_idx, DEVICE, codec,
            n_restarts=1, seed0=seed,
            n_steps=1200, lr=2e-2,
            lam_l1=5e-2, lam_bin=1e-1,
            target_keep_frac=f,
            verbose=False
        )
        m = outs[0]["mask_cross"]

        # Evaluate learned soft mask
        mae_soft, err_coeff_soft = eval_mask(skein_model, pd_list, m, DEVICE, coeff_idx=coeff_idx)

        # Evaluate hard top-k mask (more interpretable)
        topk = torch.topk(m, k=k).indices
        m_hard = torch.zeros_like(m)
        m_hard[topk] = 1.0
        mae_hard, err_coeff_hard = eval_mask(skein_model, pd_list, m_hard, DEVICE, coeff_idx=coeff_idx)

        # Random baseline (hard masks)
        rand_maes = []
        rand_errs = []
        for r in range(n_rand):
            m_r = random_binary_mask(len(m), k, seed=seed*1000 + r)
            mae_r, err_r = eval_mask(skein_model, pd_list, m_r, DEVICE, coeff_idx=coeff_idx)
            rand_maes.append(mae_r); rand_errs.append(err_r)

        results.append({
            "keep_frac": f,
            "k": k,
            "soft_full_mae": mae_soft,
            "soft_coeff_err": err_coeff_soft,
            "hard_full_mae": mae_hard,
            "hard_coeff_err": err_coeff_hard,
            "rand_full_mae_mean": float(np.mean(rand_maes)),
            "rand_full_mae_std": float(np.std(rand_maes)),
            "rand_coeff_err_mean": float(np.mean(rand_errs)),
            "rand_coeff_err_std": float(np.std(rand_errs)),
        })

    return pd.DataFrame(results)

def aggregate_mask_sweep_over_knots(
    kd,
    coeff_idx=1,                 # z^2 in folded
    knot_indices=None,           # list of indices into kd["val_df"]; if None sample
    n_knots=30,
    fracs=(0.1,0.2,0.3,0.4,0.6,0.8),
    n_rand=30,
    seed=0
):
    rng = np.random.default_rng(seed)

    if knot_indices is None:
        knot_indices = rng.choice(len(kd["val_df"]), size=min(n_knots, len(kd["val_df"])), replace=False).tolist()
    else:
        knot_indices = list(knot_indices)[:n_knots]

    per_knot_rows = []
    fails = 0

    for si in tqdm(knot_indices, desc="knots"):
        pd_list = kd["val_df"].iloc[int(si)]["pd_list"]
        if pd_list is None or len(pd_list) == 0:
            fails += 1
            continue

        try:
            df_curve = sweep_keep_frac(
                pd_list, coeff_idx,
                fracs=fracs, n_rand=n_rand, seed=seed + 10000*int(si)
            )
        except Exception as e:
            fails += 1
            continue

        df_curve = df_curve.copy()
        df_curve["knot_idx"] = int(si)

        # improvement factors (random_mean / learned_hard)
        df_curve["impr_coeff"] = df_curve["rand_coeff_err_mean"] / np.maximum(df_curve["hard_coeff_err"], 1e-9)
        df_curve["impr_full"]  = df_curve["rand_full_mae_mean"] / np.maximum(df_curve["hard_full_mae"], 1e-9)

        # win indicators (learned better than random mean)
        df_curve["win_coeff"] = (df_curve["hard_coeff_err"] < df_curve["rand_coeff_err_mean"]).astype(int)
        df_curve["win_full"]  = (df_curve["hard_full_mae"]  < df_curve["rand_full_mae_mean"]).astype(int)

        per_knot_rows.append(df_curve)

    if not per_knot_rows:
        raise RuntimeError("No knots succeeded. Check sweeper / explainer dependencies.")

    all_df = pd.concat(per_knot_rows, ignore_index=True)

    # Aggregate summary per keep_frac
    grp = all_df.groupby("keep_frac")
    summary = grp.agg(
        k=("k", "first"),
        n_knots=("knot_idx", "nunique"),

        hard_coeff_err_median=("hard_coeff_err", "median"),
        rand_coeff_err_mean=("rand_coeff_err_mean", "mean"),
        win_coeff_rate=("win_coeff", "mean"),
        impr_coeff_median=("impr_coeff", "median"),
        impr_coeff_p25=("impr_coeff", lambda x: float(np.quantile(x, 0.25))),
        impr_coeff_p75=("impr_coeff", lambda x: float(np.quantile(x, 0.75))),

        hard_full_mae_median=("hard_full_mae", "median"),
        rand_full_mae_mean=("rand_full_mae_mean", "mean"),
        win_full_rate=("win_full", "mean"),
        impr_full_median=("impr_full", "median"),
        impr_full_p25=("impr_full", lambda x: float(np.quantile(x, 0.25))),
        impr_full_p75=("impr_full", lambda x: float(np.quantile(x, 0.75))),
    ).reset_index()

    return summary, all_df

summary, all_df = aggregate_mask_sweep_over_knots(kd, coeff_idx=1, n_knots=15, fracs=(0.1,0.2,0.3,0.4,0.6,0.8), n_rand=10, seed=0)
display(summary)

In [ ]:
import numpy as np
import pandas as pd
from utils import smooth_crossing
from seifert import conway_from_pd_snappy

def model_pred_folded(pd_list):
    g = build_graph_for_inference(pd_list).to(DEVICE)
    skein_model.eval()
    with torch.no_grad():
        logits, vraw = skein_model(g)
        p = torch.sigmoid(logits)
        v = decode_values(vraw, value_mode="classification", codec=codec, decode_mode="expected")
        return (p * v).squeeze(0).detach().cpu().numpy()  # shape [L_folded]

def dict_to_folded_even(d, L_folded):
    """d maps even degree -> coeff, returns folded vector for degrees 0,2,...,2(L-1)."""
    vec = np.zeros(L_folded, dtype=np.float32)
    for deg, c in d.items():
        if deg % 2 != 0:
            continue
        j = deg // 2
        if 0 <= j < L_folded:
            vec[j] = float(c)
    return vec

def z_times_conway_from_k0(pd_k0, n_components, L_folded):
    """
    Build RHS vector for z * nabla (K0).
    """
    if n_components != 2:
        return None

    Q_even = conway_from_pd_snappy(pd_k0)         # dict even degrees (interpreted as Q(z^2))
    Q_vec  = dict_to_folded_even(Q_even, L_folded)  # Q(z^2) in folded-even

    rhs = np.zeros(L_folded, dtype=np.float32)
    # z^2*Q(z^2): coeff at z^(2j) equals coeff of Q at z^(2(j-1))
    for j in range(1, L_folded):
        rhs[j] = Q_vec[j-1]
    rhs[0] = 0.0
    return rhs

def skein_residual_test_snappy(
    kd,
    L_folded,
    n_knots=50,
    max_crossings_per_knot=5,
    seed=0,
):
    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(kd["val_df"]), size=min(n_knots, len(kd["val_df"])), replace=False)

    rows = []
    fails = {"smooth":0, "k0poly":0, "ncomp_not2":0}

    for si in idxs:
        pd_plus = kd["val_df"].iloc[int(si)]["pd_list"]
        n = len(pd_plus)
        if n == 0:
            continue

        y_plus = model_pred_folded(pd_plus)

        # choose crossings to test
        picks = rng.choice(n, size=min(max_crossings_per_knot, n), replace=False)
        for ci in picks:
            ci = int(ci)
            pd_minus = flip_crossing_pd(pd_plus, ci)
            y_minus = model_pred_folded(pd_minus)
            dy_pred = y_plus - y_minus

            try:
                pd_k0, ncomp0 = smooth_crossing(pd_plus, ci)
                if pd_k0 is None:
                    fails["smooth"] += 1
                    continue
            except Exception:
                fails["smooth"] += 1
                continue

            rhs = None
            try:
                rhs = z_times_conway_from_k0(pd_k0, ncomp0, L_folded)
            except Exception:
                fails["k0poly"] += 1
                continue

            if rhs is None:
                fails["ncomp_not2"] += 1
                continue

            resid = dy_pred - rhs

            rows.append({
                "knot_idx": int(si),
                "crossing_idx": ci,
                "ncomp_K0": int(ncomp0),
                "dy_l2": float(np.linalg.norm(dy_pred)),
                "rhs_l2": float(np.linalg.norm(rhs)),
                "resid_l2": float(np.linalg.norm(resid)),
                "rel_resid": float(np.linalg.norm(resid) / max(np.linalg.norm(rhs), 1e-6)),
                "dy0": float(dy_pred[0]),
            })

    df = pd.DataFrame(rows)
    print("Examples:", len(df))
    print("Fails:", fails)
    if len(df):
        print(df[["resid_l2","rhs_l2","rel_resid","dy0"]].describe())
    return df


In [64]:
df_skein = skein_residual_test_snappy(kd, L_folded, n_knots=80, max_crossings_per_knot=5, seed=0)
df_skein.head()

Examples: 400
Fails: {'smooth': 0, 'k0poly': 0, 'ncomp_not2': 0}
         resid_l2      rhs_l2     rel_resid           dy0
count  400.000000  400.000000  4.000000e+02  4.000000e+02
mean     5.424604    2.835766  1.495693e+05 -6.109476e-08
std      4.755332    2.535152  5.726594e+05  4.796648e-07
min      0.209222    0.000000  4.875938e-01 -4.291534e-06
25%      2.387573    1.414214  1.272556e+00  0.000000e+00
50%      3.717910    2.236068  1.850830e+00  0.000000e+00
75%      6.944955    3.464102  2.821202e+00  0.000000e+00
max     27.748810   19.646883  5.667557e+06  0.000000e+00


,knot_idx,crossing_idx,ncomp_K0,dy_l2,rhs_l2,resid_l2,rel_resid,dy0
0,909,11,2,3.396933,1.000000,2.744315,2.744315,0.0
1,909,8,2,2.793740,2.828427,2.122247,0.750328,0.0
2,909,4,2,4.259514,1.000000,4.539613,4.539613,0.0
3,909,6,2,4.989270,3.741657,3.805961,1.017186,0.0
4,909,10,2,3.465788,1.000000,2.804973,2.804973,0.0


In [68]:
df_skein.columns

Index(['knot_idx', 'crossing_idx', 'ncomp_K0', 'dy_l2', 'rhs_l2', 'resid_l2',
       'rel_resid', 'dy0'],
      dtype='object')

In [69]:
import numpy as np
import pandas as pd

def skein_correlations(df, dy_vec_col=None, rhs_vec_col=None,
                       dy_l2_col="dy_l2", rhs_l2_col="rhs_l2",
                       n_null=500, seed=0):
    rng = np.random.default_rng(seed)

    dy_norm = df[dy_l2_col].to_numpy(dtype=float)
    rhs_norm = df[rhs_l2_col].to_numpy(dtype=float)

    if dy_norm is not None and rhs_norm is not None:
        corr_norm = float(np.corrcoef(dy_norm, rhs_norm)[0, 1]) if (np.std(dy_norm) > 1e-12 and np.std(rhs_norm) > 1e-12) else float("nan")
        print(f"corr(||Delta y hat||, ||RHS||) = {corr_norm:.4f}")

        # null by shuffling RHS norms
        null = []
        for _ in range(n_null):
            perm = rng.permutation(len(rhs_norm))
            null.append(float(np.corrcoef(dy_norm, rhs_norm[perm])[0, 1]))
        null = np.array(null)
        z = (corr_norm - null.mean()) / max(null.std(), 1e-12)
        print(f"  null mean±std = {null.mean():.4f} ± {null.std():.4f}  | z={z:.1f}")

skein_correlations(df_skein, dy_vec_col="dy_vec", rhs_vec_col="rhs_vec", n_null=300, seed=0)

corr(||Delta y hat||, ||RHS||) = 0.6335
  null mean±std = 0.0010 ± 0.0509  | z=12.4
